In [1]:
# ============================================================
# Libraries
# ============================================================

import pandas as pd
import numpy as np

In [2]:
# =================================================================================================
# Function : extract_tics_from_excel
# Purpose : extract tic beginning & end from a specified time window (phase) in a single Excel file
# =================================================================================================

def extract_tics_from_excel(excel_file, phase_start_s, phase_end_s, min_absence_frames=30):

    """
    ----------
    Purpose
    ----------
    Extract tic intervals from a specified time window (phase).

    ----------
    Parameters
    ----------
    excel_file : str
        Path to the Excel annotation file.
    phase_start_s : float
        Start time of the phase to analyze (seconds).
    phase_end_s : float
        End time of the phase to analyze (seconds).
    min_absence_frames : int
        Minimum number of consecutive "Absence = 1" frames before AND after a tic.

    ----------
    Returns
    ----------
    tics : list of tuples
        [(start_time_s, end_time_s), ...] only inside the selected phase.
    """

    # load the Excel
    df = pd.read_excel(excel_file)
    # print(df.columns)


    # check the required columns
    time_col = "Time (30 fps) s"
    if time_col not in df.columns:
        raise ValueError(f"Missing required column: {time_col}")
    if "Absence" not in df.columns:
        raise ValueError("Missing required column 'Absence'.")
    times = df[time_col].values
    
    absence = df["Absence"].values

    # define explicitly the columns corresponding to the tics bodyparts
    movement_cols = ["Epaules", "Main - Bras", "Tête", "Yeux", "Visage", "Bassin - Tronc", "Jambes - Pieds", "Phonique", "Vocaux"]
    # convert into a numpy array
    movement_data = df[movement_cols].values
    # count the number of active bodyparts columns per frame
    movement_sum = movement_data.sum(axis=1)

    # tic frames : Absence = 0 & movement activity = 1+
    # is_tic_frame = (absence == 0) & (movement_sum > 0)

    tics = []
    n = len(df)
    i = 0


    # -------------------- main detection loop --------------------

    # iterate through all frames until the end
    while i < n:
        
        # to check the indices that are a problem
        # if i > 13400:
        #     print(i)
        

        # -------------------- TIC CONDITION 1 : 30 frames of "Absence" before --------------------
        before_ok = False
        before_ok = np.all((absence[i-min_absence_frames:i] == 1) & (movement_sum[i-min_absence_frames:i] == 0)) & (i >= min_absence_frames)

        if before_ok and absence[i] == 0 and movement_sum[i] > 0:
            
            start_idx = i
            start_time = times[start_idx]

            # search the end while in the condition 2
            j = i
            after_len = 0

            while after_len < min_absence_frames:
                
                while j < n and (absence[j] == 0 and movement_sum[j] > 0):
                    j += 1 # move forward frame by frame

                end_idx = j - 1
                end_time_candidate = times[end_idx]


                # -------------------- TIC CONDITION 3 : 30 frames of "Absence" after --------------------
                
                # measure how many consecutive frames are pure absence
                k = j
                while k < n and (absence[k] == 1) and (movement_sum[k] == 0):
                    k += 1
                after_len = k - j

                # to check the indices that are a problem
                # if i > 13400:
                #     print(f"J value : {j}\nK value : {k}")
                
                # verify if the lenght is enough
                after_ok = after_len >= min_absence_frames

                if after_ok:
                    # add & save the tic interval as a (start, end) pair
                    tics.append((start_time, end_time_candidate))
                    print(f"Candidate selected line : {i}")
                    i = k
                    break # and restart detection from there
                else :
                    j = k
        else :
            i += 1
    
    # filter tics inside phase
    tics_in_phase = []
    for start, end in tics:
        # keep the tic if it intersects the phase
        if end >= phase_start_s and start <= phase_end_s:
            tics_in_phase.append((start, end))

    # return all detected tic intervals
    return tics_in_phase

In [4]:
xlsx_file = ["C:\\Users\\indira.lavocat\\MOVIDOC\\PATIENT FILES\\EXCEL PATIENT FILES\\BB28_annotations_binary-table_cutted.xlsx"]

phase_start = 345.972
phase_end = 970.167
    
try:
    tics = extract_tics_from_excel(excel_file=xlsx_file, phase_start_s=phase_start, phase_end_s=phase_end, min_absence_frames=30)
    print("\nTics detected inside selected phase:")
    if not tics:
        print(" No tics detected in this phase.")
    else:
        for start, end in tics:
            print(f" - Tic from {start:.3f}s to {end:.3f}s")
except Exception as e:
    print(f"Error: {e}")

Error: Invalid file path or buffer object type: <class 'list'>


In [ ]:
# ==================================================================================================
# Function : extract_tics_from_multiple_excels
# Purpose : extract tic beginning & end from a specified time window (phase) in multiple Excel files
# ==================================================================================================

def extract_tics_from_multiple_excels(excel_files, phase_start_s, phase_end_s, min_absence_frames=30):

    """
    ----------
    Purpose
    ----------
    Run the extract_tics_from_excel function on a list of Excel files, to extract tic intervals from a specified time window (phase) in multiple Excel files

    ----------
    Parameters
    ----------
    excel_file : str
        Path to the Excel annotation file.
    phase_start_s : float
        Start time of the phase to analyze (seconds).
    phase_end_s : float
        End time of the phase to analyze (seconds).
    min_absence_frames : int
        Minimum number of consecutive "Absence = 1" frames before AND after a tic.

    ----------
    Returns
    ----------
    results :
        dict : {filename : list_of_tics}
    """

    results = {}

    for file in excel_files:
        try:
            tics = extract_tics_from_excel(excel_file=file, phase_start_s=phase_start_s, phase_end_s=phase_end_s, min_absence_frames=min_absence_frames)
            results[file] = tics
        except Exception as e:
            results[file] = f"ERROR: {e}"

    return results

In [ ]:
xlsx_files = [
    "C:\\Users\\indira.lavocat\\MOVIDOC\\PATIENT FILES\\EXCEL PATIENT FILES\\BB28_annotations_binary-table_cutted.xlsx", #BB28
    "C:\\Users\\indira.lavocat\\MOVIDOC\\PATIENT FILES\\EXCEL PATIENT FILES\\BC29_annotations_binary-table_cutted_xlsx_Lizbeth.xlsx", #BC29
    "C:\\Users\\indira.lavocat\\MOVIDOC\\PATIENT FILES\\EXCEL PATIENT FILES\\SC31_annotations_binary-table_cutted.xlsx" #SC31
]

phase_start = 345.972
phase_end = 970.167

results = extract_tics_from_multiple_excels(xlsx_files, phase_start, phase_end, min_absence_frames=30)

for file, tics in results.items():
    print(f"\nTics for {file}:")
    for start, end in tics:
        print(f" - Tic from {start:.3f}s to {end:.3f}s")